# Notebook 1: MLP for Regression from Scratch (NumPy)

In this notebook, we will build a Multi-Layer Perceptron (MLP) from scratch using only `numpy`.
Our goal is to understand the internal mechanics of a neural network: forward pass, backpropagation, activation functions, loss calculation, and optimization algorithms.

## Task: Predicting Distance from Redshift

We will use a toy dataset inspired by astrophysics: predicting the distance ($D$) of a galaxy based on its redshift ($z$).
The underlying relationship we simulate is roughly quadratic, mimicking a simplified cosmological expansion model:

$$ D \approx H_0^{-1} z + \text{higher order terms} $$

We will add some Gaussian noise to simulate measurement errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

# 1. Generate Data
def generate_data(n_samples=200):
    z = np.random.uniform(0, 3, n_samples)
    # D ~ 3000*z + 500*z^2 + noise
    noise = np.random.normal(0, 500, n_samples)
    D = 3000 * z + 500 * z**2 + noise
    
    return z.reshape(-1, 1), D.reshape(-1, 1)

X, y = generate_data()

# Normalize data for easier training (Standard Scaling)
X_mean, X_std = X.mean(), X.std()
y_mean, y_std = y.mean(), y.std()

X_norm = (X - X_mean) / X_std
y_norm = (y - y_mean) / y_std

plt.figure(figsize=(8, 5))
plt.scatter(X, y, s=10, label='Data')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('Toy Dataset: Redshift vs Distance')
plt.legend()
plt.show()

## 1. Building Blocks of a Neural Network

A neural network consists of layers. Each layer performs a linear transformation followed by a non-linear activation.

### Linear Layer

The linear layer computes:
$$ Z = X W + b $$

Where:
- $X$ is the input matrix of shape $(N, D_{in})$.
- $W$ is the weight matrix of shape $(D_{in}, D_{out})$.
- $b$ is the bias vector of shape $(1, D_{out})$.
- $Z$ is the output of shape $(N, D_{out})$.

During backpropagation, we need to compute gradients with respect to $X$, $W$, and $b$ given the gradient of the loss with respect to $Z$ ($dZ$).

$$ dX = dZ \cdot W^T $$
$$ dW = X^T \cdot dZ $$
$$ db = \sum dZ $$

In [ ]:
class Layer:
    def forward(self, input):
        pass
    def backward(self, output_gradient, learning_rate):
        pass

class Linear(Layer):
    def __init__(self, input_dim, output_dim):
        # Xavier initialization
        self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2. / input_dim)
        self.bias = np.zeros((1, output_dim))
        
        # Storage for gradients (needed for optimizers)
        self.grad_weights = None
        self.grad_bias = None

    def forward(self, input):
        self.input = input
        return np.dot(self.input, self.weights) + self.bias

    def backward(self, output_gradient):
        # Compute gradients
        self.grad_weights = np.dot(self.input.T, output_gradient)
        self.grad_bias = np.sum(output_gradient, axis=0, keepdims=True)
        
        # Return gradient w.r.t input to pass to previous layer
        return np.dot(output_gradient, self.weights.T)

### Activation Functions

We will implement **ReLU**, **LeakyReLU**, and **Sigmoid**.

**ReLU (Rectified Linear Unit):**
$$ f(x) = \max(0, x) $$
$$ f'(x) = 1 \text{ if } x > 0, \text{ else } 0 $$

**LeakyReLU:**
$$ f(x) = \max(\alpha x, x) \quad (\text{usually } \alpha=0.01) $$

**Sigmoid:**
$$ f(x) = \frac{1}{1 + e^{-x}} $$
$$ f'(x) = f(x) (1 - f(x)) $$

In [ ]:
class Activation(Layer):
    def __init__(self, activation, activation_prime):
        self.activation = activation
        self.activation_prime = activation_prime

    def forward(self, input):
        self.input = input
        return self.activation(self.input)

    def backward(self, output_gradient):
        return output_gradient * self.activation_prime(self.input)

class ReLU(Activation):
    def __init__(self):
        relu = lambda x: np.maximum(0, x)
        relu_prime = lambda x: (x > 0).astype(float)
        super().__init__(relu, relu_prime)

class LeakyReLU(Activation):
    def __init__(self, alpha=0.01):
        self.alpha = alpha
        leaky_relu = lambda x: np.where(x > 0, x, x * alpha)
        leaky_relu_prime = lambda x: np.where(x > 0, 1, alpha)
        super().__init__(leaky_relu, leaky_relu_prime)

class Sigmoid(Activation):
    def __init__(self):
        def sigmoid(x):
            return 1 / (1 + np.exp(-x))
        def sigmoid_prime(x):
            s = sigmoid(x)
            return s * (1 - s)
        super().__init__(sigmoid, sigmoid_prime)

### Loss Function with Regularization

We use Mean Squared Error (MSE) for regression:
$$ L = \frac{1}{N} \sum (y_{pred} - y_{true})^2 $$

**Regularization (L1/L2):**
We can add a penalty to the loss to prevent overfitting.

- **L2 (Ridge):** $\lambda \sum w^2$
- **L1 (Lasso):** $\lambda \sum |w|$

The gradients for the weights need to be updated accordingly during the optimization step, or we can add the regularization gradient to the backward pass.

In [ ]:
class MSE:
    def forward(self, pred, true):
        return np.mean((pred - true) ** 2)

    def backward(self, pred, true):
        return 2 * (pred - true) / true.size

## 2. Optimizers

Gradient descent algorithms update the weights to minimize the loss.

### SGD (Stochastic Gradient Descent)
$$ W = W - \eta \cdot \nabla W $$

### Momentum
Accumulates a velocity vector:
$$ v_t = \gamma v_{t-1} + \eta \nabla W $$
$$ W = W - v_t $$

### RMSProp
Scales the learning rate by a running average of the squared magnitudes of the gradients.

### Adam (Adaptive Moment Estimation)
Combines Momentum and RMSProp. It computes adaptive learning rates for each parameter.
$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t $$
$$ v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 $$
$$ \hat{m}_t = m_t / (1 - \beta_1^t) $$
$$ \hat{v}_t = v_t / (1 - \beta_2^t) $$
$$ W = W - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} $$

In [ ]:
class Optimizer:
    def step(self, layers):
        pass

class SGD(Optimizer):
    def __init__(self, learning_rate=0.01):
        self.lr = learning_rate

    def step(self, layers):
        for layer in layers:
            if not isinstance(layer, Linear): continue
                      
            # Update weights
            layer.weights -= self.lr * layer.grad_weights
            layer.bias -= self.lr * layer.grad_bias

class Adam(Optimizer):
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}
        self.v = {}
        self.t = 0

    def step(self, layers):
        self.t += 1
        for layer in layers:
            if not isinstance(layer, Linear): continue
            
            if layer not in self.m:
                self.m[layer] = {'w': np.zeros_like(layer.weights), 'b': np.zeros_like(layer.bias)}
                self.v[layer] = {'w': np.zeros_like(layer.weights), 'b': np.zeros_like(layer.bias)}
            
            grad_w = layer.grad_weights
            grad_b = layer.grad_bias

            # Update first moment (m)
            self.m[layer]['w'] = self.beta1 * self.m[layer]['w'] + (1 - self.beta1) * grad_w
            self.m[layer]['b'] = self.beta1 * self.m[layer]['b'] + (1 - self.beta1) * grad_b

            # Update second moment (v)
            self.v[layer]['w'] = self.beta2 * self.v[layer]['w'] + (1 - self.beta2) * grad_w**2
            self.v[layer]['b'] = self.beta2 * self.v[layer]['b'] + (1 - self.beta2) * grad_b**2

            # Bias correction
            m_hat_w = self.m[layer]['w'] / (1 - self.beta1**self.t)
            v_hat_w = self.v[layer]['w'] / (1 - self.beta2**self.t)
            m_hat_b = self.m[layer]['b'] / (1 - self.beta1**self.t)
            v_hat_b = self.v[layer]['b'] / (1 - self.beta2**self.t)

            # Update parameters
            layer.weights -= self.lr * m_hat_w / (np.sqrt(v_hat_w) + self.epsilon)
            layer.bias -= self.lr * m_hat_b / (np.sqrt(v_hat_b) + self.epsilon)

## 3. Training the Model

We will build a simple network: `Input -> Linear -> LeakyReLU -> Linear -> Output`.

In [ ]:
# Network Architecture
network = [
    Linear(1, 16),
    LeakyReLU(),
    Linear(16, 16),
    LeakyReLU(),
    Linear(16, 1)
]

loss_fn = MSE()
# Try switching between SGD and Adam
# optimizer = SGD(learning_rate=0.01, momentum=0.9, l2_lambda=0.001)
optimizer = Adam(learning_rate=0.01)

# Training Loop
epochs = 2000
loss_history = []

for epoch in range(epochs):
    # Forward Pass
    output = X_norm
    for layer in network:
        output = layer.forward(output)
    
    # Loss
    loss = loss_fn.forward(output, y_norm)
    loss_history.append(loss)
    
    # Backward Pass
    grad = loss_fn.backward(output, y_norm)
    for layer in reversed(network):
        grad = layer.backward(grad)
    
    # Optimization Step
    optimizer.step(network)
    
    if epoch % 200 == 0:
        print(f'Epoch {epoch}, Loss: {loss:.5f}')

# Visualize Loss
plt.plot(loss_history)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.show()

In [ ]:
# Predictions
output = X_norm
for layer in network:
    output = layer.forward(output)

# Denormalize predictions
preds = output * y_std + y_mean

plt.figure(figsize=(8, 5))
plt.scatter(X, y, s=10, label='Data', color='gray', alpha=0.5)
plt.scatter(X, preds, s=10, label='Prediction', color='red')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('MLP Prediction vs True Data')
plt.legend()
plt.show()